# 13: Homology Generators from Small Simplicial Complexes

This notebook shows how to find and visualize homology generators on five tiny examples (each under 20 vertices).
The simplicial complexes are built with GUDHI, and the homology-generator extraction is routed through the Julia bridge when available.

We focus on three ingredients: 
- `compute_homology_basis_from_simplex_tree` for `H_0` and `H_2`,
- `compute_optimal_h1_basis_from_simplex_tree` for cycle representatives in `H_1`,
- `HomologyGenerator` objects, which expose `support_edges` and `support_simplices`.

The examples are designed to highlight different generator patterns:
1. many connected components plus one loop,
2. two disjoint loops,
3. a sphere boundary with non-trivial `H_2`,
4. two disconnected sphere boundaries,
5. a triangulated torus with non-trivial `H_1` and `H_2`.

## Imports

In [1]:
import numpy as np
import plotly.graph_objects as go
import gudhi

from pysurgery.core.generator_models import HomologyGenerator
from pysurgery.core.complexes import CWComplex
from pysurgery.core.fundamental_group import (
    FundamentalGroup,
    extract_pi_1_with_traces,
    infer_standard_group_descriptor,
)
from pysurgery.core.homology_generators import (
    compute_homology_basis_from_simplex_tree,
    compute_optimal_h1_basis_from_simplex_tree,
)
from pysurgery.bridge.julia_bridge import julia_engine
from pysurgery.integrations.gudhi_bridge import extract_complex_data

print('HomologyGenerator model:', HomologyGenerator.__name__)

HomologyGenerator model: HomologyGenerator


## Julia bridge check

The bridge is used here to validate that the notebook is exercising the Julia-backed path.

In [2]:
print("Julia available:", julia_engine.available)
if julia_engine.available:
    warmup_report = julia_engine.warmup()
    print("Julia warm-up completed.")
    print("  completed workloads:", len(warmup_report.get("completed", [])))
    print("  pi1 warm-up included:", "pi1_abelianization" in warmup_report.get("completed", []))
    if warmup_report.get("failed"):
        print("  failed workloads:", sorted(warmup_report["failed"].keys()))
else:
    print("Julia backend is unavailable; homology generators will use the Python fallback path.")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Julia available: True
Julia warm-up completed.
  completed workloads: 14
  pi1 warm-up included: True
Julia warm-up completed.
  completed workloads: 14
  pi1 warm-up included: True


In [3]:
# PI1 generator mode selector: change to "raw" for the full spanning-forest presentation.
PI1_GENERATOR_MODE = "optimized"
print("PI1 generator mode selector:", PI1_GENERATOR_MODE)


PI1 generator mode selector: optimized
PI1 generator mode selector: optimized


## Helper functions

We build simplex trees directly from simplices, then extract and plot generators from the resulting `HomologyBasisResult` objects.

In [8]:
def build_simplex_tree(num_vertices, simplices):
    st = gudhi.SimplexTree()
    for v in range(num_vertices):
        st.insert([v])
    for simplex in simplices:
        st.insert(list(simplex))
    return st


def summarize_basis(label, basis):
    print(f"{label}: rank={basis.rank}, exact={basis.exact}, optimal={basis.optimal}")
    print(f"  message: {basis.message}")
    for i, gen in enumerate(basis.generators[:5]):
        print(
            f"  generator {i}: dim={gen.dimension}, "+
            f"edges={len(gen.support_edges)}, simplices={len(gen.support_simplices)}"
        )


def _first_h0_vertices(h0_basis, max_points=8):
    verts = []
    for gen in h0_basis.generators:
        for simplex in gen.support_simplices:
            if len(simplex) == 1:
                verts.append(int(simplex[0]))
    seen = set()
    out = []
    for v in verts:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out[:max_points]


def _edges_from_h1_basis(h1_basis, max_generators=4):
    return [[tuple(sorted(edge)) for edge in gen.support_edges] for gen in h1_basis.generators[:max_generators]]


def _faces_from_h2_basis(h2_basis, max_faces=200):
    faces = []
    seen = set()
    for gen in h2_basis.generators:
        for simplex in gen.support_simplices:
            if len(simplex) == 3:
                face = tuple(sorted(simplex))
                if face not in seen:
                    seen.add(face)
                    faces.append(face)
                if len(faces) >= max_faces:
                    return faces
    return faces


def _simplex_tree_to_cw_complex(st):
    boundaries, cells, _, _ = extract_complex_data(st)
    return CWComplex(cells=cells, attaching_maps=boundaries, coefficient_ring="Z")


def _pi1_trace_paths(pi1_with_traces, max_traces=None):
    out = []
    traces = pi1_with_traces.traces if max_traces is None else pi1_with_traces.traces[:max_traces]
    for tr in traces:
        out.append((tr.generator, [(int(a), int(b)) for a, b in tr.undirected_edge_path]))
    return out


def plot_pi1_simplices(title, st, pts, pi1_with_traces, max_traces=None):
    """Plot only the simplices that compose pi1 generator traces."""
    edges = [tuple(s) for s, _ in st.get_skeleton(1) if len(s) == 2]
    fig = go.Figure()

    if edges:
        xs, ys, zs = [], [], []
        for u, v in edges:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(width=2, color='lightgray'),
            name='1-skeleton background', showlegend=True,
        ))

    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers+text',
        text=[str(i) for i in range(len(pts))],
        textposition='top center',
        marker=dict(size=5, color='royalblue'),
        name='vertices (indexed simplices)', showlegend=True,
    ))

    colors = ['black', 'red', 'orange', 'purple', 'green', 'magenta', 'cyan']
    for gi, (gen_name, edge_path) in enumerate(_pi1_trace_paths(pi1_with_traces, max_traces=max_traces)):
        xs, ys, zs = [], [], []
        for u, v in edge_path:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(width=7, color=colors[gi % len(colors)]),
            name=f'pi1 generator {gen_name}', showlegend=True,
        ))

    fig.update_layout(
        title=f"{title} — Fundamental Group Generator Simplices",
        height=760, width=1200,
        showlegend=True,
        legend=dict(x=0.0, y=1.0, bgcolor='rgba(255,255,255,0.85)'),
    )
    return fig


def plot_complex(title, st, pts, h0_basis, h1_basis, h2_basis, pi1_with_traces=None):
    faces = [tuple(s) for s, _ in st.get_skeleton(2) if len(s) == 3]
    edges = [tuple(s) for s, _ in st.get_skeleton(1) if len(s) == 2]

    fig = go.Figure()

    if faces:
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[f[0] for f in faces], j=[f[1] for f in faces], k=[f[2] for f in faces],
            opacity=0.22, color='lightblue', name='2-skeleton mesh', showlegend=True,
        ))

    if edges:
        xs, ys, zs = [], [], []
        for u, v in edges:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(width=3, color='dimgray'),
            name='1-skeleton edges', showlegend=True,
        ))

    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers', marker=dict(size=5, color='royalblue', symbol='circle'),
        name='0-skeleton vertices', showlegend=True,
    ))

    h0_vs = _first_h0_vertices(h0_basis)
    if h0_vs:
        fig.add_trace(go.Scatter3d(
            x=pts[h0_vs, 0], y=pts[h0_vs, 1], z=pts[h0_vs, 2],
            mode='markers', marker=dict(size=9, color='black', symbol='diamond'),
            name='H0 generators', showlegend=True,
        ))

    colors = ['red', 'orange', 'purple', 'green', 'magenta', 'cyan']
    for gi, edge_list in enumerate(_edges_from_h1_basis(h1_basis, max_generators=4)):
        xs, ys, zs = [], [], []
        for u, v in edge_list:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(width=6, color=colors[gi % len(colors)]),
            name=f'H1 generator {gi}', showlegend=True,
        ))

    h2_faces = _faces_from_h2_basis(h2_basis)
    if h2_faces:
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[f[0] for f in h2_faces], j=[f[1] for f in h2_faces], k=[f[2] for f in h2_faces],
            opacity=0.55, color='gold', name='H2 generator support', showlegend=True,
        ))

    if pi1_with_traces is not None:
        for gen_name, edge_path in _pi1_trace_paths(pi1_with_traces, max_traces=None):
            xs, ys, zs = [], [], []
            for u, v in edge_path:
                xs.extend([pts[u, 0], pts[v, 0], None])
                ys.extend([pts[u, 1], pts[v, 1], None])
                zs.extend([pts[u, 2], pts[v, 2], None])
            fig.add_trace(go.Scatter3d(
                x=xs, y=ys, z=zs, mode='lines',
                line=dict(width=4, color='black', dash='dash'),
                name=f'pi1 trace {gen_name}', showlegend=True,
            ))

    fig.update_layout(
        title=title,
        height=760, width=1200,
        showlegend=True,
        legend=dict(x=0.0, y=1.0, bgcolor='rgba(255,255,255,0.85)'),
    )
    return fig


def analyze_example(name, st, pts, generator_mode=PI1_GENERATOR_MODE):
    print(f'\n=== {name} ===')
    boundaries, bridge_cells, _, _ = extract_complex_data(st)
    print(f"Julia bridge cell dimensions: {sorted(bridge_cells.keys())}")

    cw = CWComplex(cells=bridge_cells, attaching_maps=boundaries, coefficient_ring="Z")
    pi1_raw = extract_pi_1_with_traces(cw, simplify=True, generator_mode="raw")
    pi1_opt = extract_pi_1_with_traces(cw, simplify=True, generator_mode="optimized")
    pi1_with_traces = extract_pi_1_with_traces(cw, simplify=True, generator_mode=generator_mode)
    print(
        "pi_1 mode comparison: "
        f"raw={pi1_raw.raw_generator_count} generators, "
        f"optimized={pi1_opt.optimized_generator_count} generators"
    )
    print(
        f"pi_1 selected mode: requested={generator_mode}, "
        f"used={pi1_with_traces.mode_used}, backend={pi1_with_traces.backend_used}"
    )
    pi1_raw_group = FundamentalGroup(generators=list(pi1_raw.generators), relations=[list(r) for r in pi1_raw.relations])
    pi1_opt_group = FundamentalGroup(generators=list(pi1_opt.generators), relations=[list(r) for r in pi1_opt.relations])
    pi1 = FundamentalGroup(generators=list(pi1_with_traces.generators), relations=[list(r) for r in pi1_with_traces.relations])

    raw_descriptor = infer_standard_group_descriptor(pi1_raw_group)
    opt_descriptor = infer_standard_group_descriptor(pi1_opt_group)
    selected_descriptor = infer_standard_group_descriptor(pi1)

    raw_descriptor = raw_descriptor if raw_descriptor is not None else "not certified"
    opt_descriptor = opt_descriptor if opt_descriptor is not None else "not certified"
    selected_descriptor = selected_descriptor if selected_descriptor is not None else "not certified"

    print(f"pi_1 generators={pi1.generators}, relations={pi1.relations}")
    print(
        "pi_1 certified canonical descriptor (selected mode): "
        f"{selected_descriptor}"
    )
    print(
        "pi_1 descriptor by mode: "
        f"raw={raw_descriptor}, optimized={opt_descriptor}"
    )
    if pi1_with_traces.traces:
        print("pi_1 generator simplices (edge traces):")
        for tr in pi1_with_traces.traces:
            print(f"  {tr.generator}: edges={tr.undirected_edge_path}, vertices={tr.vertex_path}")
    else:
        print("pi_1 generator simplices: []")

    h0 = compute_homology_basis_from_simplex_tree(st, dimension=0)
    h1 = compute_optimal_h1_basis_from_simplex_tree(st, point_cloud=pts, max_cycles=8)
    h2 = compute_homology_basis_from_simplex_tree(st, dimension=2, mode='valid')

    summarize_basis('H0', h0)
    summarize_basis('H1', h1)
    summarize_basis('H2', h2)

    fig = plot_complex(name, st, pts, h0, h1, h2, pi1_with_traces=pi1_with_traces)
    fig.show()

    fig_pi1 = plot_pi1_simplices(name, st, pts, pi1_with_traces)
    fig_pi1.show()

    return {'name': name, 'h0': h0, 'h1': h1, 'h2': h2, 'pi1': pi1_with_traces}


## Example builders

All examples use at most 16 vertices. The goal is to make the generators easy to read visually and algebraically.

In [9]:
def build_three_isolated_points_and_triangle_loop():
    pts = np.array([
        [-4.0, 0.0, 0.0],
        [-2.5, 1.0, 0.0],
        [-2.0, -1.2, 0.0],
        [ 2.0, 0.0, 0.0],
        [ 3.4, 1.2, 0.0],
        [ 3.0, -1.1, 0.0],
    ], dtype=float)
    simplices = [(3, 4), (4, 5), (5, 3)]
    return build_simplex_tree(6, simplices), pts


def build_two_triangle_loops():
    pts = np.array([
        [-4.0, 0.0, 0.0],
        [-2.8, 1.2, 0.0],
        [-2.2, -1.1, 0.0],
        [ 2.0, 0.0, 0.0],
        [ 3.3, 1.2, 0.0],
        [ 3.7, -1.0, 0.0],
    ], dtype=float)
    simplices = [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)]
    return build_simplex_tree(6, simplices), pts


def build_tetrahedron_boundary():
    pts = np.array([
        [0.0, 0.0, 0.0],
        [1.4, 0.0, 0.0],
        [0.7, 1.2, 0.0],
        [0.7, 0.4, 1.2],
    ], dtype=float)
    simplices = [
        (0, 1, 2),
        (0, 1, 3),
        (0, 2, 3),
        (1, 2, 3),
    ]
    return build_simplex_tree(4, simplices), pts


def build_two_tetrahedron_boundaries():
    pts_a = np.array([
        [0.0, 0.0, 0.0],
        [1.4, 0.0, 0.0],
        [0.7, 1.2, 0.0],
        [0.7, 0.4, 1.2],
    ], dtype=float)
    pts_b = pts_a + np.array([4.0, 0.0, 0.0], dtype=float)
    pts = np.vstack([pts_a, pts_b])
    simplices = [
        (0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3),
        (4, 5, 6), (4, 5, 7), (4, 6, 7), (5, 6, 7),
    ]
    return build_simplex_tree(8, simplices), pts


def build_torus_4x4(R=3.0, r=1.0):
    nu = nv = 4
    pts = []
    simplices = []

    def idx(i, j):
        return i * nv + j

    for i in range(nu):
        u = 2.0 * np.pi * i / nu
        for j in range(nv):
            v = 2.0 * np.pi * j / nv
            x = (R + r * np.cos(v)) * np.cos(u)
            y = (R + r * np.cos(v)) * np.sin(u)
            z = r * np.sin(v)
            pts.append([x, y, z])

    for i in range(nu):
        ip = (i + 1) % nu
        for j in range(nv):
            jp = (j + 1) % nv
            a = idx(i, j)
            b = idx(ip, j)
            c = idx(ip, jp)
            d = idx(i, jp)
            simplices.append((a, b, c))
            simplices.append((a, c, d))

    return build_simplex_tree(16, simplices), np.array(pts, dtype=float)


## Five worked examples

Expected ranks:

| Example | H0 | H1 | H2 |
|---|---:|---:|---:|
| 1. Three isolated points + triangle loop | 4 | 1 | 0 |
| 2. Two disjoint triangle loops | 2 | 2 | 0 |
| 3. Tetrahedron boundary | 1 | 0 | 1 |
| 4. Two tetrahedron boundaries | 2 | 0 | 2 |
| 5. Torus (4x4 triangulation) | 1 | 2 | 1 |

In [10]:
examples = [
    ('Example 1 — three isolated points + one loop', build_three_isolated_points_and_triangle_loop()),
    ('Example 2 — two disjoint triangle loops', build_two_triangle_loops()),
    ('Example 3 — boundary of a tetrahedron', build_tetrahedron_boundary()),
    ('Example 4 — two disconnected tetrahedron boundaries', build_two_tetrahedron_boundaries()),
    ('Example 5 — triangulated torus (4x4)', build_torus_4x4()),
]

results = []
for title, (st, pts) in examples:
    results.append(analyze_example(title, st, pts))


=== Example 1 — three isolated points + one loop ===
Julia bridge cell dimensions: [0, 1]
pi_1 mode comparison: raw=1 generators, optimized=1 generators
pi_1 selected mode: requested=optimized, used=optimized, backend=julia
pi_1 generators=['g_2'], relations=[]
pi_1 certified canonical descriptor (selected mode): Z
pi_1 descriptor by mode: raw=Z, optimized=Z
pi_1 generator simplices (edge traces):
  g_2: edges=[(4, 5), (3, 5), (3, 4)], vertices=[4, 5, 3, 4]
H0: rank=4, exact=True, optimal=False
  message: Computed H_0 representatives over Z/2 via Julia backend.
  generator 0: dim=0, edges=0, simplices=1
  generator 1: dim=0, edges=0, simplices=1
  generator 2: dim=0, edges=0, simplices=1
  generator 3: dim=0, edges=0, simplices=1
H1: rank=1, exact=True, optimal=True
  message: Computed by shortest-path generator set + greedy independent basis over Z2 annotations (Julia backend).
  generator 0: dim=1, edges=4, simplices=4
H2: rank=0, exact=True, optimal=False
  message: Computed H_2 re


=== Example 2 — two disjoint triangle loops ===
Julia bridge cell dimensions: [0, 1]
pi_1 mode comparison: raw=2 generators, optimized=2 generators
pi_1 selected mode: requested=optimized, used=optimized, backend=julia
pi_1 generators=['g_2', 'g_5'], relations=[]
pi_1 certified canonical descriptor (selected mode): not certified
pi_1 descriptor by mode: raw=not certified, optimized=not certified
pi_1 generator simplices (edge traces):
  g_2: edges=[(1, 2), (0, 2), (0, 1)], vertices=[1, 2, 0, 1]
  g_5: edges=[(4, 5), (3, 5), (3, 4)], vertices=[4, 5, 3, 4]
H0: rank=2, exact=True, optimal=False
  message: Computed H_0 representatives over Z/2 via Julia backend.
  generator 0: dim=0, edges=0, simplices=1
  generator 1: dim=0, edges=0, simplices=1
H1: rank=2, exact=True, optimal=True
  message: Computed by shortest-path generator set + greedy independent basis over Z2 annotations (Julia backend).
  generator 0: dim=1, edges=4, simplices=4
  generator 1: dim=1, edges=4, simplices=4
H2: rank


=== Example 3 — boundary of a tetrahedron ===
Julia bridge cell dimensions: [0, 1, 2]
pi_1 mode comparison: raw=3 generators, optimized=0 generators
pi_1 selected mode: requested=optimized, used=optimized, backend=julia
pi_1 generators=[], relations=[]
pi_1 certified canonical descriptor (selected mode): 1
pi_1 descriptor by mode: raw=1, optimized=1
pi_1 generator simplices: []
H0: rank=1, exact=True, optimal=False
  message: Computed H_0 representatives over Z/2 via Julia backend.
  generator 0: dim=0, edges=0, simplices=1
H1: rank=0, exact=True, optimal=True
  message: Computed by shortest-path generator set + greedy independent basis over Z2 annotations (Julia backend).
H2: rank=1, exact=True, optimal=False
  message: Computed H_2 representatives over Z/2 via Julia backend.
  generator 0: dim=2, edges=0, simplices=4



=== Example 4 — two disconnected tetrahedron boundaries ===
Julia bridge cell dimensions: [0, 1, 2]
pi_1 mode comparison: raw=6 generators, optimized=0 generators
pi_1 selected mode: requested=optimized, used=optimized, backend=julia
pi_1 generators=[], relations=[]
pi_1 certified canonical descriptor (selected mode): 1
pi_1 descriptor by mode: raw=1, optimized=1
pi_1 generator simplices: []
H0: rank=2, exact=True, optimal=False
  message: Computed H_0 representatives over Z/2 via Julia backend.
  generator 0: dim=0, edges=0, simplices=1
  generator 1: dim=0, edges=0, simplices=1
H1: rank=0, exact=True, optimal=True
  message: Computed by shortest-path generator set + greedy independent basis over Z2 annotations (Julia backend).
H2: rank=2, exact=True, optimal=False
  message: Computed H_2 representatives over Z/2 via Julia backend.
  generator 0: dim=2, edges=0, simplices=4
  generator 1: dim=2, edges=0, simplices=4



=== Example 5 — triangulated torus (4x4) ===
Julia bridge cell dimensions: [0, 1, 2]
pi_1 mode comparison: raw=33 generators, optimized=2 generators
pi_1 selected mode: requested=optimized, used=optimized, backend=julia
pi_1 generators=['g_41', 'g_46'], relations=[['g_41', 'g_46', 'g_41^-1', 'g_46^-1']]
pi_1 certified canonical descriptor (selected mode): Z x Z
pi_1 descriptor by mode: raw=Z x Z, optimized=Z x Z
pi_1 generator simplices (edge traces):
  g_41: edges=[(10, 15), (0, 15), (0, 5), (5, 10)], vertices=[10, 15, 0, 5, 10]
  g_46: edges=[(13, 14), (3, 14), (0, 3), (0, 1), (1, 13)], vertices=[13, 14, 3, 0, 1, 13]
H0: rank=1, exact=True, optimal=False
  message: Computed H_0 representatives over Z/2 via Julia backend.
  generator 0: dim=0, edges=0, simplices=1
H1: rank=2, exact=True, optimal=True
  message: Computed by shortest-path generator set + greedy independent basis over Z2 annotations (Julia backend).
  generator 0: dim=1, edges=5, simplices=5
  generator 1: dim=1, edges=

## Takeaways

- `H_0` generators pick one representative vertex per connected component.
- `H_1` generators are edge cycles. In these examples, the basis is returned as `HomologyGenerator` objects with `support_edges`.
- `H_2` generators are collections of faces that close up into a 2-cycle.
- The same API works for disconnected graphs, sphere boundaries, and a torus.
- `infer_standard_group_descriptor` now reports a certified canonical `pi_1` descriptor when provable (for this torus example: `Z x Z`).

If you want to inspect a generator programmatically, look at `basis.generators[i].support_edges` and `basis.generators[i].support_simplices`.